# Assignment 6 — Generation Experiment

**DATAX504** · Due end of Week 11

Choose **text** or **image** generation. This notebook is intentionally sparse: pick a small, runnable experiment you can defend in the reflection.

## Tasks
1. Train or adapt a small generator (char-RNN / small LM, GAN, VAE, simple diffusion demo, …)
2. Show **5 samples** in this notebook
3. Name the main **failure mode** you observed
4. After reading the [HF diffusion language models post](https://huggingface.co/blog/ProCreations/diffusion-language-model), write ~150 words on AR vs diffusion trade-offs

## Moodle fields
`a6_repo`, `a6_modality`, `a6_failure`, `a6_diffusion_reflection`

## AI disclosure (required)
Fill the table in the next cell before you submit.


## AI disclosure

| Field | Your answer |
|-------|-------------|
| Tool / model | |
| Used for | |
| Verified how | |
| Not used for | |


In [ ]:
MODALITY = "text"  # — Moodle a6_modality

import numpy as np
import matplotlib.pyplot as plt
from keras import layers, Sequential, optimizers, losses, callbacks
from datasets import load_dataset

print("Selected modality:", MODALITY)

## Experiment

Implement **one** modality below (remove or ignore the other).

Keep the scope small enough to train on a laptop. Document data source, training setup, and how you sampled outputs.


In [ ]:
SEQ_LEN = 64        # For data split
STRIDE = 32
TRAIN_SIZE = 10000   # Decrease training time
VAL_SIZE = 2000
BATCH_SIZE = 512    # For model training
EPOCHS = 20
LENGTH = 200        # For generating output
TEMPERATURE = 0.3
SEEDS = [           # For results
    "ROMEO:\n", 
    "First Citizen:\n", 
    "KING HENRY:\n", 
    "JULIET:\n", 
    "HAMLET:\n"
] 

# data loading + preprocessing 
def load_shakespeare():
    ds = load_dataset("Trelis/tiny-shakespeare")
    col = [c for c, f in ds["train"].features.items() if f.dtype == "string"][0]
    print("Text column", col)
    return ds, col

def get_text(ds, col):
    text = "\n".join(ds["train"][col])
    val_text = "\n".join(ds["test"][col])
    print(f"Train chars: {len(text)}, Val chars: {len(val_text)}")
    return text, val_text

def get_chars(text, val_text):
    chars = sorted(set(text + val_text))
    char2idx = {c: i for i, c in enumerate(chars)}
    idx2char = np.array(chars)
    vocab_size = len(chars)
    return chars, char2idx, idx2char, vocab_size

def create_x_y(s, char2idx, seq_len=SEQ_LEN, stride=STRIDE):
    data = np.array([char2idx[c] for c in s], dtype=np.int32)
    starts = np.arange(0, len(data) - seq_len - 1, stride)
    x = np.stack([data[i: i + seq_len] for i in starts])
    y = np.stack([data[i + 1: i + seq_len + 1] for i in starts])
    return x, y

def split_ds(text, val_text, char2idx):
    x_train, y_train = create_x_y(text, char2idx)
    x_train, y_train = subset_train(x_train, y_train)
    x_val, y_val = create_x_y(val_text, char2idx)
    x_val, y_val = subset_val(x_val, y_val)
    print(f"Train seqs: {x_train.shape}, Val seqs: {x_val.shape}")
    return x_train, y_train, x_val, y_val

def subset_train(x_train, y_train, train_size=TRAIN_SIZE):
    return x_train[:train_size], y_train[:train_size]

def subset_val(x_val, y_val, val_size=VAL_SIZE):
    return x_val[:val_size], y_val[:val_size]

# model definition
def create_model():
    return Sequential([
        layers.Input(shape=(None,), dtype=np.int32),
        layers.Embedding(vocab_size, 64),
        layers.GRU(128, return_sequences=True),
        layers.Dropout(0.2),
        layers.GRU(128, return_sequences=True),
        layers.Dense(vocab_size)        
    ])

def compile_model(model):
    model.compile(
        optimizer=optimizers.Adam(0.05),
        loss=losses.SparseCategoricalCrossentropy(from_logits=True)
    )
    model.summary()

# training loop
def get_callbacks():
    return [
        callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2
        ),
        callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True
        )
    ]

def train_model(model, x_train, y_train, x_val, y_val, batch_size=BATCH_SIZE, epochs=EPOCHS):
    return model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        batch_size=batch_size,
        epochs=epochs,
        callbacks=get_callbacks()
    )
    
def plot_training(history):
    plt.plot(history.history["loss"], label="train")
    plt.plot(history.history["val_loss"], label="val")
    plt.xlabel("epochs")
    plt.ylabel("loss")
    plt.legend()
    plt.show()
    
# generate and display 5 samples
def generate(seed, length=LENGTH, temperature=TEMPERATURE):
    out = seed
    for _ in range(length):
        ctx = [char2idx.get(c, 0) for c in out[-SEQ_LEN:]]
        logits = np.asarray(model(np.array([ctx]), training=False))[0, -1]
        logits = logits / temperature
        probs = np.exp(logits - logits.max())
        probs /= probs.sum()
        out += idx2char[np.random.choice(vocab_size, p=probs)]
    return out

def print_result():
    for i, s in enumerate(SEEDS, 1):
        print(f"Sample {i} (seed={s.strip()!r}, T={TEMPERATURE})")
        print(generate(s))
        print()

if MODALITY == "text":
    # Data loading and preprocessing
    ds, col = load_shakespeare()
    text, val_text = get_text(ds, col)
    chars, char2idx, idx2char, vocab_size = get_chars(text, val_text)
    x_train, y_train, x_val, y_val = split_ds(text, val_text, char2idx)
    
    # Model definition
    model = create_model()
    compile_model(model)
    
    # Training loop
    history = train_model(model, x_train, y_train, x_val, y_val)
    plot_training(history)
    
    # Generate 5 samples
    print_result()
    
elif MODALITY == "image":
    pass  # your image experiment
else:
    raise ValueError("MODALITY must be 'text' or 'image'")


## Failure mode (Moodle `a6_failure`)

One short phrase is enough for Moodle (e.g. repetition, mode collapse, blur). Expand briefly below if useful.


*Failure mode observed:*


## AR vs diffusion reflection (Moodle `a6_diffusion_reflection`)

~150 words after the HF reading: compare autoregressive vs diffusion-style generation for *your* modality (quality, control, compute, failure modes).


*Your reflection (~150 words).*
